# AcessaCidade — ELT em camadas Raw → Bronze → Silver → Gold
### Disciplina 02 · Big Data Architecture & Data Integration · FIAP Global Solution 2026.2 (CitizenTech, ODS 11)

> **Crédito:** este notebook adapta a didática do notebook da **Aula 06 — "De ETL para ELT: Camadas Raw e Bronze"**
> (Unidade III · Ingestão, Integração e Orquestração de Dados) da disciplina Big Data Architecture & Data Integration
> (FIAP). Reaproveitamos as ideias da aula — ELT dentro do DuckDB, RAW fiel com metadados `_source_file` /
> `_ingested_at` / `_row_id`, leitura `all_varchar` (schema-on-read), funções de parsing Python registradas no DuckDB
> com `create_function`, `TRY_CAST`, dedup técnico com `row_number() OVER (PARTITION BY <todas as colunas>)`, flags
> de qualidade, separação técnico × negócio e DAG TaskFlow (`@dag`/`@task`) — trocando o domínio de **vendas** pelo
> domínio de **denúncias de acessibilidade** do AcessaCidade e indo além da Bronze, até **Silver** e **Gold**.

**Diferença importante em relação à aula:** aqui o notebook **não contém a lógica do pipeline**. Ele chama as funções
do pacote `acessacidade_pipeline` (pasta `02-Big-Data-Architecture/src`) — exatamente as mesmas que a DAG do Airflow
e o runner local (`python -m acessacidade_pipeline`) executam. Assim há uma única implementação, testada por `pytest`.

| Camada | O que contém | O que NÃO contém |
|---|---|---|
| **raw** | Cópia fiel das fontes (tudo texto) + metadados de linhagem | Nenhuma transformação |
| **bronze** | Tipos corretos, duplicatas técnicas removidas, problemas **sinalizados** em `flag_*`, nome do denunciante removido (LGPD) | Regras de negócio |
| **silver** | Regras de negócio RN-01..RN-12: bairros padronizados, chaves do contrato, dedup de negócio, `registro_valido` | Agregações |
| **gold** | Agregados prontos para o dashboard/BI + `gold_snapshot.json` (contrato 4.8) | Dados pessoais e texto livre |

> **Todos os dados usados aqui são SIMULADOS** (cidade fictícia "Cidade Exemplo", nomes do tipo "Pessoa Simulada 017").

## 0. Preparação

Tornamos o pacote importável (`../src`) e lemos a configuração. Os caminhos vêm de variáveis de ambiente
(`PIPELINE_DATA_DIR`, `ACESSACIDADE_API_URL`, `PIPELINE_TOKEN`); sem elas, usamos a pasta `data/` do repositório e
o warehouse DuckDB em `data/processed/warehouse/acessacidade.duckdb`.

In [1]:
import sys, json, logging
from pathlib import Path

SRC = Path.cwd().parent / "src"          # notebook roda em 02-Big-Data-Architecture/notebooks
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 40)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", stream=sys.stdout, force=True)

from acessacidade_pipeline import __version__, parsing, regras, simulate
from acessacidade_pipeline import run_pipeline as rp
from acessacidade_pipeline.config import PipelineConfig, REPO_ROOT
from acessacidade_pipeline.warehouse import conectar

cfg = PipelineConfig.from_env()
cfg.garantir_pastas()
def rel(p):
    try: return str(Path(p).resolve().relative_to(REPO_ROOT)).replace("\\", "/")
    except ValueError: return str(p)
print("pacote acessacidade_pipeline", __version__)
print("CSV simulado :", rel(cfg.samples_csv))
print("warehouse    :", rel(cfg.warehouse_path))
print("API configurada?", cfg.api_configurada)

def sql(q):
    # Consulta somente leitura no warehouse (cada tarefa do pipeline abre e fecha a sua conexão).
    with conectar(cfg, read_only=True) as con:
        return con.execute(q).df()

pacote acessacidade_pipeline 1.0.0
CSV simulado : data/samples/denuncias_simuladas.csv
warehouse    : data/processed/warehouse/acessacidade.duckdb
API configurada? False


## 1. As fontes

1. **`data/samples/denuncias_simuladas.csv`** — exportação "bagunçada" de um sistema legado **fictício** da prefeitura
   fictícia da Cidade Exemplo, gerada por `simulate.py` com semente fixa (sempre o mesmo arquivo). Tem as colunas da
   exportação da API (contrato 4.7) **mais** `nome_denunciante` (para demonstrar minimização LGPD) e `origem_dado=SIMULADO`.
2. **Exportação da API do AcessaCidade** — `GET {ACESSACIDADE_API_URL}/api/export/denuncias?formato=csv` com o header
   `X-Pipeline-Token`. Neste notebook rodamos **sem a API** (`usar_api=False`); a extração real é testada com
   `httpx.MockTransport` em `tests/test_pipeline_api.py` e acontece na DAG quando a API e o token estão configurados.

Vamos olhar a fonte simulada como texto puro, do jeito que chega:

In [2]:
bruto = pd.read_csv(cfg.samples_csv, dtype=str, keep_default_na=False)
print(f"{len(bruto)} linhas x {bruto.shape[1]} colunas | origem_dado: {bruto.origem_dado.unique().tolist()}")
bruto[["id", "criado_em", "bairro", "tipo_obstaculo", "status", "latitude", "longitude", "nome_denunciante"]].head(8)

271 linhas x 23 colunas | origem_dado: ['SIMULADO']


,id,criado_em,bairro,tipo_obstaculo,status,latitude,longitude,nome_denunciante
0,213,2026-09-20T00:51:02Z,bela vista,obra sem sinalização,EM_ANALISE,,,Pessoa Simulada 060
1,84,27 de outubro de 2026,Parque das Árvores,calçada quebrada,Em Analise,-10.157506,"-48,322707",Pessoa Simulada 102
2,203,2026-08-27 07:04:51,JARDIM DAS FLORES,buraco_calcada,aguardando revisao,,,Pessoa Simulada 056
3,130,21/09/2026 05:36,são josé,obra sem sinalização,aguardando revisao,,,
4,113,2026-08-24 04:34:19,santa luzia,BURACO_CALCADA,,,,
5,88,2026-10-12 20:30:18,CENTRO,obra sem sinalização,em analise,,,Pessoa Simulada 064
6,188,29/08/2026 21:17,CENTRO,Buraco na calcada,,,,Pessoa Simulada 061
7,106,2026-11-12T10:47:01Z,Nova Esperança,BURACO_VIA,Em Analise,,,Pessoa Simulada 120


Mesmo dado, muitas grafias. Exemplo: quantas formas diferentes de escrever bairro, tipo de obstáculo e status?

In [3]:
for col in ["bairro", "tipo_obstaculo", "impacto", "status", "visao_classificacao"]:
    print(f"{col:20s}: {bruto[col].nunique():3d} grafias distintas, ex.: {bruto[col].value_counts().index[:5].tolist()}")
print("linhas idênticas repetidas:", int(bruto.duplicated().sum()))

bairro              :  42 grafias distintas, ex.: ['Jd das Flores', 'Centro ', 'Bela  Vista', 'Centro', 'Parque das Árvores']
tipo_obstaculo      :  38 grafias distintas, ex.: ['piso solto na calçada', 'calçada quebrada', 'buraco calcada', 'CALCADA_DANIFICADA', 'Buraco na calcada']
impacto             :  16 grafias distintas, ex.: ['DIFICULTA_PASSAGEM', 'dificulta', 'risco de queda', 'impede passagem', 'Dificulta a passagem']
status              :  13 grafias distintas, ex.: ['Aguardando revisão', 'Em Analise', 'aguardando revisao', 'EM_ANALISE', 'Em análise']
visao_classificacao :  10 grafias distintas, ex.: ['', 'Nenhum indício suficiente de buraco', 'Análise inconclusiva', 'NOT_IDENTIFIED', 'possible_pothole']
linhas idênticas repetidas: 11


## 2. Funções de parsing (parte TÉCNICA) — e por que elas vivem no Python

Como na aula, as regras de parsing são funções Python pequenas e testáveis (`parsing.py`), **registradas no DuckDB**
com `con.create_function(...)` para serem usadas dentro do SQL. Veja o que elas fazem com valores reais da fonte:

In [4]:
exemplos_data = ["2026-10-27T22:11:20Z", "12/08/2026", "08-08-2026 00:24", "14 de outubro de 2026",
                 "31/09/2026", "ontem", ""]
pd.DataFrame({"texto": exemplos_data, "parse_data_multi (UTC)": [parsing.parse_data_multi(v) for v in exemplos_data]})

,texto,parse_data_multi (UTC)
0,2026-10-27T22:11:20Z,2026-10-27 22:11:20
1,12/08/2026,2026-08-12 03:00:00
2,08-08-2026 00:24,2026-08-08 03:24:00
3,14 de outubro de 2026,2026-10-14 03:00:00
4,31/09/2026,NaT
5,ontem,NaT
6,,NaT


In [5]:
exemplos_coord = ["-48,322707", " -10.157506 ", "abc", "-10.18.2", "-123.4", ""]
pd.DataFrame({"texto": exemplos_coord, "parse_coordenada": [parsing.parse_coordenada(v) for v in exemplos_coord]})

,texto,parse_coordenada
0,"-48,322707",-48.322707
1,-10.157506,-10.157506
2,abc,NaN
3,-10.18.2,NaN
4,-123.4,-123.400000
5,,NaN


In [6]:
from acessacidade_pipeline.parsing import FUNCOES_SQL as F_TECNICAS
from acessacidade_pipeline.regras import FUNCOES_SQL as F_NEGOCIO
print("funções técnicas registradas no DuckDB :", [f[0] for f in F_TECNICAS])
print("funções de negócio registradas no DuckDB:", [f[0] for f in F_NEGOCIO])

funções técnicas registradas no DuckDB : ['parse_data_multi', 'parse_coordenada', 'parse_decimal', 'parse_booleano', 'normalizar_texto', 'limpar_espacos']
funções de negócio registradas no DuckDB: ['padronizar_bairro', 'bairro_reconhecido', 'padronizar_cidade', 'normalizar_logradouro', 'normalizar_numero', 'mapear_tipo_obstaculo', 'tipo_obstaculo_reconhecido', 'mapear_impacto', 'mapear_status', 'mapear_visao', 'mapear_origem', 'mapear_revisao']


## 3. Extract + Load → camada RAW

`tarefa_extract_and_load_raw` lê as fontes e grava **exatamente como chegaram**: todas as colunas como `VARCHAR`
(`read_csv(..., all_varchar = true)`), sem limpeza, sem dedup — apenas com metadados de linhagem
`_source_file`, `_ingested_at`, `_row_id`, `_batch_id`. A carga é *full refresh* (as fontes são exportações completas),
o que torna o pipeline **idempotente**.

In [7]:
estado = rp.tarefa_extract_and_load_raw(execucao_id="notebook__demonstracao", usar_api=False, cfg=cfg)
{k: estado[k] for k in ("execucao_id", "linhas_raw_simuladas", "linhas_raw_app", "linhas_raw")}

INFO [extract] execução notebook__demonstracao | configuração: {'data_dir': 'data', 'samples_csv': 'data/samples/denuncias_simuladas.csv', 'exports_dir': 'data/exports', 'gold_dir': 'data/processed/gold', 'warehouse_path': 'data/processed/warehouse/acessacidade.duckdb', 'api_url': '(não configurada)', 'pipeline_token': '(não configurado)'}


WARNING [extract] API PULADA: Execução com --sem-api: somente a fonte simulada.


INFO [raw] raw.denuncias_simuladas: 271 linhas | raw.denuncias_app: 0 linhas


{'execucao_id': 'notebook__demonstracao',
 'linhas_raw_simuladas': 271,
 'linhas_raw_app': 0,
 'linhas_raw': 271}

In [8]:
sql("SELECT _row_id, _source_file, _ingested_at, _batch_id, criado_em, bairro, latitude, nome_denunciante "
    "FROM raw.denuncias_simuladas ORDER BY _row_id LIMIT 5")

,_row_id,_source_file,_ingested_at,_batch_id,criado_em,bairro,latitude,nome_denunciante
0,1,data/samples/denuncias_simuladas.csv,2026-11-15 06:26:58.976779,notebook__demonstracao,2026-09-20T00:51:02Z,bela vista,None,Pessoa Simulada 060
1,2,data/samples/denuncias_simuladas.csv,2026-11-15 06:26:58.976779,notebook__demonstracao,27 de outubro de 2026,Parque das Árvores,-10.157506,Pessoa Simulada 102
2,3,data/samples/denuncias_simuladas.csv,2026-11-15 06:26:58.976779,notebook__demonstracao,2026-08-27 07:04:51,JARDIM DAS FLORES,None,Pessoa Simulada 056
3,4,data/samples/denuncias_simuladas.csv,2026-11-15 06:26:58.976779,notebook__demonstracao,21/09/2026 05:36,são josé,None,None
4,5,data/samples/denuncias_simuladas.csv,2026-11-15 06:26:58.976779,notebook__demonstracao,2026-08-24 04:34:19,santa luzia,None,None


In [9]:
# RAW fiel: mesma quantidade de linhas do arquivo e tudo texto
print("linhas no arquivo:", len(bruto), "| linhas na RAW:", sql("SELECT count(*) n FROM raw.denuncias_simuladas").n[0])
sql("SELECT DISTINCT data_type FROM information_schema.columns WHERE table_schema='raw' AND table_name='denuncias_simuladas' AND column_name NOT LIKE '\\_%' ESCAPE '\\'")

linhas no arquivo:

 271 | linhas na RAW: 271


,data_type
0,VARCHAR


In [10]:
sql("SELECT * FROM raw.log_cargas")

,_batch_id,fonte,tabela,arquivo,linhas,status,mensagem,carregado_em
0,manual__2026-11-15T06:16:46,denuncias_simuladas.csv,raw.denuncias_simuladas,data/samples/denuncias_simuladas.csv,271,OK,Fonte simulada carregada sem transfo...,2026-11-15 06:16:46.053654
1,manual__2026-11-15T06:16:46,api_export,raw.denuncias_app,None,0,PULADA,Execução com --sem-api: somente a fo...,2026-11-15 06:16:46.053654
2,notebook__demonstracao,denuncias_simuladas.csv,raw.denuncias_simuladas,data/samples/denuncias_simuladas.csv,271,OK,Fonte simulada carregada sem transfo...,2026-11-15 06:26:58.976779
3,notebook__demonstracao,api_export,raw.denuncias_app,None,0,PULADA,Execução com --sem-api: somente a fo...,2026-11-15 06:26:58.976779


## 4. Transform (técnico) → camada BRONZE

`tarefa_transform_bronze` une as duas fontes RAW (coluna `fonte`) e:
- remove **duplicatas técnicas** — linha idêntica campo a campo: `row_number() OVER (PARTITION BY <todas as colunas de origem>)`;
- **tipa**: datas → `TIMESTAMP` UTC (`parse_data_multi`), coordenadas e pontuação → `DOUBLE`, booleanos → `BOOLEAN`,
  id → `BIGINT` com `TRY_CAST` (valor ruim vira `NULL`, a carga não cai);
- **sinaliza** problemas em `flag_*` em vez de corrigi-los;
- **minimização LGPD**: `nome_denunciante` não passa da RAW — fica só `flag_possui_nome`.

In [11]:
estado = rp.tarefa_transform_bronze(estado, cfg=cfg)
{k: v for k, v in estado.items() if "bronze" in k or "dup" in k}

INFO [bronze] {'linhas_raw': 271, 'linhas_bronze': 260, 'duplicatas_tecnicas_simuladas': 11, 'duplicatas_tecnicas_app': 0, 'duplicatas_removidas': 11}


{'linhas_bronze': 260,
 'duplicatas_tecnicas_simuladas': 11,
 'duplicatas_tecnicas_app': 0,
 'duplicatas_removidas': 11}

In [12]:
sql('''SELECT count(*) AS linhas,
       sum(flag_data_invalida::INT) AS data_invalida, sum(flag_data_ausente::INT) AS data_ausente,
       sum(flag_coordenada_invalida::INT) AS coord_invalida, sum(flag_coordenada_ausente::INT) AS coord_ausente,
       sum(flag_campo_obrigatorio_ausente::INT) AS obrigatorio_ausente, sum(flag_possui_nome::INT) AS tinha_nome
FROM bronze.denuncias''')

,linhas,data_invalida,data_ausente,coord_invalida,coord_ausente,obrigatorio_ausente,tinha_nome
0,260,11.0,1.0,8.0,116.0,25.0,155.0


In [13]:
# A coluna de nome NÃO existe na Bronze (nem em nenhuma camada depois dela)
sql("SELECT table_schema, table_name, column_name FROM information_schema.columns WHERE column_name ILIKE '%nome%' ORDER BY 1,2")

,table_schema,table_name,column_name
0,bronze,denuncias,flag_possui_nome
1,raw,denuncias_simuladas,nome_denunciante
2,silver,denuncias,flag_possui_nome
3,silver,denuncias_duplicadas,flag_possui_nome


In [14]:
sql("SELECT fonte, _row_id, criado_em, latitude, longitude, flag_coordenada_invalida, flag_data_invalida FROM bronze.denuncias "
    "WHERE flag_coordenada_invalida OR flag_data_invalida ORDER BY _row_id LIMIT 8")

,fonte,_row_id,criado_em,latitude,longitude,flag_coordenada_invalida,flag_data_invalida
0,denuncias_simuladas.csv,10,2026-10-07 16:03:56,NaN,NaN,True,False
1,denuncias_simuladas.csv,25,NaT,-10.187658,-48.318430,False,True
2,denuncias_simuladas.csv,35,NaT,NaN,NaN,False,True
3,denuncias_simuladas.csv,37,NaT,-10.157064,-48.292155,False,True
4,denuncias_simuladas.csv,67,2026-10-15 08:43:00,NaN,NaN,True,False
5,denuncias_simuladas.csv,69,NaT,-10.162510,-48.344203,False,True
6,denuncias_simuladas.csv,70,NaT,NaN,NaN,False,True
7,denuncias_simuladas.csv,73,2026-10-22 09:59:27,NaN,NaN,True,False


## 5. Regras de negócio → camada SILVER

Na aula 06 a Silver ficou como "próxima camada". Aqui ela existe, com regras numeradas e documentadas em
`docs/REGRAS_DE_NEGOCIO.md` (RN-01..RN-13). Destaques:
- **RN-01** bairros padronizados por dicionário (sem acento, minúsculo, abreviações `jd`→`jardim`, `pq`→`parque`...);
- **RN-04..RN-07** tipo de obstáculo, impacto, status e classificação da visão mapeados para as **chaves do contrato**;
- **RN-09/RN-10** datas plausíveis e coordenadas dentro do Brasil;
- **RN-11** dedup de negócio: mesma ocorrência = mesmo endereço normalizado + mesmo tipo + mesma data local;
- **RN-12** regra de registro válido (`registro_valido` + `motivos_invalidade`).

In [15]:
estado = rp.tarefa_curate_silver(estado, cfg=cfg)
{k: v for k, v in estado.items() if "silver" in k or "valid" in k or "negocio" in k}

INFO [silver] {'linhas_silver': 250, 'duplicatas_negocio_removidas': 10, 'registros_validos': 211, 'registros_invalidos': 39}


{'linhas_silver': 250,
 'duplicatas_negocio_removidas': 10,
 'registros_validos': 211,
 'registros_invalidos': 39}

In [16]:
sql('''SELECT bairro AS bairro_padronizado, count(DISTINCT bairro_original) AS grafias_na_fonte,
       string_agg(DISTINCT '"' || bairro_original || '"', ', ') AS exemplos
FROM silver.denuncias WHERE bairro IS NOT NULL GROUP BY 1 ORDER BY 2 DESC''')

,bairro_padronizado,grafias_na_fonte,exemplos
0,Jardim das Flores,6,"""Jardim das Flores"", ""Jd. das Flore..."
1,Vila Nova,5,"""V. Nova"", ""vila nova"", ""Vl. Nova"", ..."
2,São José,5,"""são josé"", ""SAO JOSE"", ""S. José"", ""..."
3,Bela Vista,4,"""Bela Vista"", ""bela vista"", ""BELA V..."
4,Nova Esperança,4,"""NOVA ESPERANÇA"", ""Nova Esperança"", ..."
5,Parque das Árvores,4,"""Pq. das Arvores"", ""Parque das Arvor..."
6,Santa Luzia,4,"""Santa Luzia"", ""santa luzia"", ""Sta L..."
7,Centro,3,"""CENTRO"", ""Centro"", ""centro"""
8,Recanto Verde,3,"""recanto verde"", ""Recanto Verde"", ""R..."


In [17]:
sql("SELECT tipo_obstaculo, tipo_obstaculo_rotulo, count(*) total FROM silver.denuncias GROUP BY ALL ORDER BY total DESC")

,tipo_obstaculo,tipo_obstaculo_rotulo,total
0,CALCADA_DANIFICADA,Calçada quebrada ou desnivelada,63
1,BURACO_CALCADA,Buraco na calçada,57
2,RAMPA_IRREGULAR,Rampa ausente ou irregular,35
3,OBSTACULO_TEMPORARIO,"Obstáculo temporário (entulho, veícu...",33
4,OBSTACULO_FIXO,"Obstáculo fixo (poste, lixeira, placa)",21
5,BURACO_VIA,Buraco na via ou travessia,19
6,PISO_TATIL,Piso tátil ausente ou danificado,11
7,OUTRO,Outro,9
8,None,None,2


In [18]:
sql("SELECT chave_denuncia_mantida, chave_denuncia AS removida, bairro, tipo_obstaculo, data_registro FROM silver.denuncias_duplicadas LIMIT 5")

,chave_denuncia_mantida,removida,bairro,tipo_obstaculo,data_registro
0,denuncias_simuladas.csv:000062,denuncias_simuladas.csv:000031,Bela Vista,BURACO_CALCADA,2026-09-08
1,denuncias_simuladas.csv:000087,denuncias_simuladas.csv:000074,Jardim das Flores,RAMPA_IRREGULAR,2026-10-22
2,denuncias_simuladas.csv:000110,denuncias_simuladas.csv:000090,Bela Vista,CALCADA_DANIFICADA,2026-11-07
3,denuncias_simuladas.csv:000199,denuncias_simuladas.csv:000114,Santa Luzia,BURACO_VIA,2026-10-05
4,denuncias_simuladas.csv:000022,denuncias_simuladas.csv:000116,Vila Nova,OBSTACULO_FIXO,2026-08-05


In [19]:
sql('''SELECT motivo, count(*) AS registros FROM (
         SELECT unnest(string_split(motivos_invalidade, '; ')) AS motivo FROM silver.denuncias WHERE NOT registro_valido)
       GROUP BY 1 ORDER BY 2 DESC''')

,motivo,registros
0,impacto ausente ou não reconhecido,15
1,data de registro ausente ou inválida,12
2,descrição ausente ou com menos de 10...,7
3,logradouro ausente,5
4,bairro ausente,3
5,tipo de obstáculo ausente,2


## 6. Camada GOLD — o que alimenta o produto

`tarefa_build_gold` agrega **somente registros válidos** (RN-13), sem dados pessoais nem texto livre, e exporta:
- `data/processed/gold/*.csv` (para o dashboard de BI da disciplina 03);
- `data/processed/gold/gold_snapshot.json` no formato do **contrato 4.8** — é o que a tarefa `publish_gold` envia para
  `POST /api/pipeline/gold`, e que a aplicação mostra em `GET /api/indicadores/analiticos`.

In [20]:
estado = rp.tarefa_build_gold(estado, cfg=cfg)
print("arquivos:", estado["arquivos_gold"]); print("snapshot:", estado["snapshot"])
sql("SELECT * FROM gold.denuncias_por_bairro")

INFO [gold] {'linhas_gold_denuncias_por_bairro': 9, 'linhas_gold_denuncias_por_categoria': 8, 'linhas_gold_denuncias_por_periodo': 19, 'linhas_gold_denuncias_por_status': 5, 'linhas_gold_resultado_visao': 4, 'linhas_gold_qualidade_registros': 10}


INFO [gold] arquivos exportados: denuncias_por_bairro.csv, denuncias_por_categoria.csv, denuncias_por_periodo.csv, denuncias_por_status.csv, resultado_visao.csv, qualidade_registros.csv, gold_snapshot.json


arquivos: ['data/processed/gold/denuncias_por_bairro.csv', 'data/processed/gold/denuncias_por_categoria.csv', 'data/processed/gold/denuncias_por_periodo.csv', 'data/processed/gold/denuncias_por_status.csv', 'data/processed/gold/resultado_visao.csv', 'data/processed/gold/qualidade_registros.csv', 'data/processed/gold/gold_snapshot.json']
snapshot: data/processed/gold/gold_snapshot.json


,bairro,total,possiveis_buracos,impede_passagem,com_fotografia,percentual
0,Centro,39,6,13,25,18.5
1,Jardim das Flores,36,11,11,25,17.1
2,São José,36,6,9,18,17.1
3,Bela Vista,24,5,10,15,11.4
4,Parque das Árvores,21,4,7,12,10.0
5,Vila Nova,19,2,5,11,9.0
6,Santa Luzia,18,4,5,8,8.5
7,Nova Esperança,17,2,3,10,8.1
8,Recanto Verde,1,0,0,1,0.5


In [21]:
sql("SELECT * FROM gold.denuncias_por_periodo WHERE granularidade = 'mensal'")

,granularidade,periodo,inicio_periodo,total,possiveis_buracos
0,mensal,2026-08,2026-08-01,57,12
1,mensal,2026-09,2026-09-01,66,10
2,mensal,2026-10,2026-10-01,61,11
3,mensal,2026-11,2026-11-01,27,7


In [22]:
sql("SELECT * FROM gold.resultado_visao")

,classificacao,rotulo,total,percentual
0,POSSIBLE_POTHOLE,Possível buraco identificado,40,19.0
1,NOT_IDENTIFIED,Nenhum indício suficiente de buraco,44,20.9
2,INCONCLUSIVE,Análise inconclusiva,37,17.5
3,None,Sem fotografia (não analisada),90,42.7


In [23]:
snapshot = json.loads(cfg.snapshot_path.read_text(encoding="utf-8"))
print("chaves do snapshot:", list(snapshot))
print(json.dumps({k: snapshot[k] for k in ("disponivel", "inclui_simulados", "fontes", "qualidade")}, ensure_ascii=False, indent=2))

chaves do snapshot: ['disponivel', 'gerado_em', 'execucao_id', 'inclui_simulados', 'fontes', 'denuncias_por_bairro', 'denuncias_por_categoria', 'denuncias_por_periodo', 'denuncias_por_status', 'resultado_visao', 'qualidade', 'denuncias_por_semana']
{
  "disponivel": true,
  "inclui_simulados": true,
  "fontes": [
    {
      "nome": "denuncias_simuladas.csv",
      "linhas": 271
    },
    {
      "nome": "api_export",
      "linhas": 0
    }
  ],
  "qualidade": {
    "linhas_raw": 271,
    "linhas_bronze": 260,
    "duplicatas_removidas": 11,
    "datas_invalidas": 12,
    "coordenadas_invalidas": 11,
    "campos_obrigatorios_ausentes": 25,
    "percentual_registros_validos": 84.4,
    "linhas_silver": 250,
    "duplicatas_negocio_removidas": 10,
    "registros_validos": 211
  }
}


## 7. Relatório de qualidade

`tarefa_quality_report` conta raw/bronze/silver, duplicatas e flags, roda verificações automáticas (RAW fiel à fonte,
nenhuma coluna de nome fora da RAW, snapshot no formato do contrato, somas da Gold coerentes...) e grava
`data/processed/relatorio_qualidade.json` e `.md`. Se alguma verificação falhar, a situação fica `REPROVADO`.

In [24]:
estado = rp.tarefa_quality_report(estado, cfg=cfg)
relatorio = json.loads(cfg.relatorio_json_path.read_text(encoding="utf-8"))
print("situação:", relatorio["situacao"])
pd.DataFrame(relatorio["verificacoes"])

INFO [qualidade] RAW não vazia: OK (271 linhas na RAW)


INFO [qualidade] RAW fiel à fonte simulada: OK (arquivo=271 raw=271)


INFO [qualidade] BRONZE ≤ RAW: OK (260 ≤ 271)


INFO [qualidade] SILVER + duplicadas = BRONZE: OK (260 = 260)


INFO [qualidade] Sem coluna de nome fora da RAW: OK (nenhuma)


INFO [qualidade] Arquivos Gold sem dados pessoais: OK (nenhum)


... (3 outputs left out)

INFO [qualidade] Percentual de válidos entre 0 e 100: OK (84.4%)


INFO [qualidade] situação: APROVADO | relatório: data/processed/relatorio_qualidade.json e data/processed/relatorio_qualidade.md


situação: APROVADO


,verificacao,ok,detalhe
0,RAW não vazia,True,271 linhas na RAW
1,RAW fiel à fonte simulada,True,arquivo=271 raw=271
2,BRONZE ≤ RAW,True,260 ≤ 271
3,SILVER + duplicadas = BRONZE,True,260 = 260
4,Sem coluna de nome fora da RAW,True,nenhuma
5,Arquivos Gold sem dados pessoais,True,nenhum
6,Snapshot no formato do contrato 4.8,True,ok
7,Chaves Gold pertencem ao contrato (s...,True,ok
8,Agregados Gold somam os registros vá...,True,válidos=211 somas={'por_bairro': 211...
9,Percentual de válidos entre 0 e 100,True,84.4%


## 8. Publicação e orquestração

`tarefa_publish_gold` faz `POST {ACESSACIDADE_API_URL}/api/pipeline/gold` com `X-Pipeline-Token`. Sem API/token
(como aqui), a tarefa é **pulada com mensagem clara** — o pipeline não falha por isso.

In [25]:
estado = rp.tarefa_publish_gold(estado, cfg=cfg)
estado["publicacao"]

WARNING [publish] Execução com --sem-api: publicação na API não realizada.


{'status': 'PULADA',
 'mensagem': 'Execução com --sem-api: publicação na API não realizada.'}

No Airflow, as mesmas seis funções viram seis tarefas TaskFlow em `dags/acessacidade_elt_pipeline.py`
(`schedule="@daily"`, `catchup=False`, `retries=1`):

```
extract_and_load_raw → transform_bronze → curate_silver → build_gold → quality_report → publish_gold
```

Cada tarefa abre a própria conexão com o DuckDB e devolve um dicionário pequeno (XCom) com as contagens — o mesmo
`estado` que vimos acima. Idempotência: rodar de novo recria todas as tabelas (`CREATE OR REPLACE`) e produz o mesmo
resultado (verificado em `tests/test_pipeline_idempotencia.py`).

In [26]:
print("Ordem das tarefas (runner local = DAG):", " -> ".join(rp.ORDEM_TAREFAS))
sql('''SELECT 'raw' camada, count(*) linhas FROM raw.denuncias_simuladas
       UNION ALL SELECT 'bronze', count(*) FROM bronze.denuncias
       UNION ALL SELECT 'silver', count(*) FROM silver.denuncias
       UNION ALL SELECT 'silver (válidos)', count(*) FROM silver.denuncias WHERE registro_valido''')

Ordem das tarefas (runner local = DAG): extract_and_load_raw -> transform_bronze -> curate_silver -> build_gold -> quality_report -> publish_gold


,camada,linhas
0,raw,271
1,bronze,260
2,silver,250
3,silver (válidos),211


## Resumo

| Camada | Pergunta que responde |
|---|---|
| raw | "O que exatamente chegou, de onde e quando?" |
| bronze | "Quais valores são tecnicamente utilizáveis e onde estão os problemas?" |
| silver | "Qual é a denúncia única, padronizada segundo as regras do AcessaCidade?" |
| gold | "Quais indicadores o produto e o BI mostram?" |

> Lembrete: os números acima vêm de **dados simulados** e não representam nenhuma cidade real.